# Roblox Asset AI - Specialized Model Fine-Tuning
### Kaggle T4 GPU Environment (16GB VRAM)

This notebook fine-tunes a specialized code/3D LLM (e.g. `Qwen/Qwen2.5-Coder-1.5B-Instruct` or `meta-llama/Llama-3.2-1B-Instruct`) using **LoRA (Low-Rank Adaptation)** specifically on **Roblox Asset Intermediate Representations (IR)**, animations, and self-improvement critique trajectories.

**Pipeline Trajectories Trained:**
1. `text` → Roblox 3D Model IR
2. `text + reference image` → Roblox 3D Model IR
3. `model IR + rendered image` → Vision Critique (missing parts, proportions, shape errors)
4. `model IR + critique` → Corrected/Refined Model IR
5. `text` → Roblox KeyframeSequence Animation IR

In [ ]:
# 1. Hardware Inspection: Verify NVIDIA T4 GPU
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Install Required Kaggle Packages
!pip install -q transformers datasets peft accelerate bitsandbytes

In [ ]:
# 3. Run Dataset Generation & Check Dataset Integrity
import json

dataset_path = "./roblox_assets_train.jsonl"
print("Loading dataset preview...")
with open(dataset_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < 3:
            item = json.loads(line)
            print(f"Sample {i+1} Task: {item.get('taskType')}")
            print(f"Prompt: {item.get('prompt')}")
            print(f"Asset Name: {item.get('targetAsset', {}).get('name')}")
            print("-" * 40)

In [ ]:
# 4. Launch Fine-Tuning with LoRA & Checkpoint Resumption
!python train.py \
    --model_name "Qwen/Qwen2.5-Coder-1.5B-Instruct" \
    --data_dir "." \
    --output_dir "./checkpoints/roblox-asset-ai-t4" \
    --batch_size 2 \
    --grad_accum 4 \
    --epochs 3 \
    --lr 2e-4

In [ ]:
# 5. Evaluate and Export Model Artifacts
eval_path = "./checkpoints/roblox-asset-ai-t4/eval_results.json"
with open(eval_path, "r") as f:
    results = json.load(f)
print("Training Evaluation Results:")
print(json.dumps(results, indent=2))

# Package weights for download / deployment
!tar -czvf roblox_asset_ai_weights.tar.gz ./checkpoints/roblox-asset-ai-t4